# Radar de Carreira IA — da vaga ao próximo passo

**Pergunta:** como usar anúncios reais para explorar competências e orientar um plano de aprendizado?

Este notebook acompanha o projeto Python. Os resultados são de uma amostra de conveniência do Jobicy, não de todo o mercado. Perfis usados nos exemplos são fictícios. A coleta e as vagas são reais.

Para executar, selecione o ambiente Python com as dependências do projeto. Jupyter pode ser instalado separadamente com `python -m pip install jupyterlab`. Execute as células na ordem.

In [1]:
from pathlib import Path
import sys
import json
from collections import Counter
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'radar').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from radar.data import load_snapshot
from radar.model import CareerModel
from radar.evaluation import evaluate_categories
from radar.skills import SKILLS

snapshot = load_snapshot()
df = pd.DataFrame(snapshot['jobs'])
print('Coleta UTC:', snapshot['metadata']['fetched_at'])
print('Vagas:', len(df), '| Empresas:', df.company.nunique())
print('SHA-256:', snapshot['metadata']['sha256'])

Coleta UTC: 2026-09-20T01:27:05.870291+00:00
Vagas: 316 | Empresas: 158
SHA-256: f81fbfb40d4e972e4fcc26eaf276039f5c760dcfcbace3a0e4ca9e1904555bea


## 1. Qualidade, cobertura e viés

IDs e descrições foram deduplicados. HTML foi convertido em texto. As localidades são restrições informadas pela fonte; remoto não implica contratação no Brasil. Competências são menções extraídas por regras, incluindo possíveis diferenciais e contexto.

In [2]:
print(df.category.value_counts().to_string())
print('\nVagas sem competências reconhecidas:', int(df.skills.map(len).eq(0).sum()))
print('Registros sem salário mínimo:', int(df.salary_min.isna().sum()))
print('IDs duplicados:', int(df.id.duplicated().sum()))
print('\nLocalidades mais frequentes:')
print(df.location.value_counts().head(8).to_string())

category
Engenharia de software    199
Dados e analytics         117

Vagas sem competências reconhecidas: 30
Registros sem salário mínimo: 166
IDs duplicados: 0

Localidades mais frequentes:
location
USA            114
Europe          21
Canada          20
Anywhere        14
UK              12
LATAM           11
Canada, USA      9
Poland           9


## 2. Competências em destaque

Cada competência é contada no máximo uma vez por vaga. Os percentuais usam todas as vagas como denominador e podem somar mais de 100%. Não são tendências de crescimento.

In [3]:
frequency = Counter(skill for skills in df.skills for skill in skills)
top = pd.DataFrame(frequency.most_common(15), columns=['competencia', 'vagas'])
top['percentual_da_amostra'] = (top.vagas / len(df) * 100).round(1)
print(top.to_string(index=False))

     competencia  vagas  percentual_da_amostra
          Python    135                   42.7
             SQL    135                   42.7
            APIs    105                   33.2
             AWS     92                   29.1
Machine Learning     83                   26.3
            LLMs     77                   24.4
             Git     64                   20.3
           React     53                   16.8
             ETL     51                   16.1
           CI/CD     50                   15.8
      Kubernetes     50                   15.8
     Estatística     50                   15.8
      TypeScript     49                   15.5
           Azure     48                   15.2
             GCP     43                   13.6


## 3. Recomendação e agrupamento

O TF-IDF aprende pesos de palavras e pares de palavras. A recomendação combina cosseno com cobertura ponderada por IDF. O agrupamento K-means usa vetores de competências; selecionamos o melhor silhouette entre k=2 e k=6 na própria amostra. Essa escolha é exploratória.

In [4]:
model = CareerModel(snapshot['jobs'])
print('Vocabulário textual:', len(model.vectorizer.vocabulary_))
print('Competências do dicionário:', len(SKILLS))
print(pd.DataFrame(model.cluster_trials).to_string(index=False))
print('\nGrupos:')
print(pd.DataFrame(model.cluster_info).to_string(index=False))

Vocabulário textual: 14000
Competências do dicionário: 53
 k  silhouette
 2      0.1287
 3      0.1236
 4      0.1412
 5      0.1537
 6      0.1585

Grupos:
 id  count                            name
 -1     30      Sem competências extraídas
  0     57       React · TypeScript · APIs
  1     71 Machine Learning · Python · SQL
  2     54         AWS · Kubernetes · APIs
  3     35 Análise de dados · SQL · Python
  4     34              SQL · Python · dbt
  5     35            APIs · LLMs · Python


## 4. Simular uma nova competência

Mantemos o mesmo texto e a mesma população de vagas. Só adicionamos AWS ao conjunto de competências consideradas presentes. O ganho é de cobertura média, não de salário, proficiência ou chance de contratação.

In [5]:
profile = 'Python SQL Pandas análise de dados Power BI visualização de dados'
indices = list(range(len(df)))
current = model.recommend(profile, indices)
future = model.recommend(profile, indices, extra_skills=['AWS'])
print('Atual:', current['simulation'])
print('Simulada:', future['simulation'])
print('\nPrimeiras recomendações:')
print(pd.DataFrame(future['matches'])[['title', 'company', 'score', 'coverage']].head(5).to_string(index=False))
print('\nPróximas competências por ganho marginal:')
print(pd.DataFrame(future['plan'])[['skill', 'jobs', 'gain']].to_string(index=False))

Atual: {'before': 18.0, 'after': 18.0, 'above80_before': 10, 'above80_after': 10}
Simulada: {'before': 18.0, 'after': 21.4, 'above80_before': 10, 'above80_after': 11}

Primeiras recomendações:
                                                                                                   title                  company  score  coverage
                                                               Data Analyst, Clinical Data Effectiveness            Clover Health   69.0     100.0
                                                       Associate Research Scientist, Real World Evidence Precision Medicine Group   66.2     100.0
                                      Costpoint Cognos Report Writer and General Ledger Business Analyst                  ManTech   66.2     100.0
                                                                      PRINCIPAL R&D/PRODUCT DVL ENGINEER          TE Connectivity   66.1     100.0
Projets de collecte de données/d'enregistrement | Français & Anglais | R

## 5. Experimento supervisionado separado

Alvo: categoria atribuída pelo Jobicy. Modelos: baseline de classe majoritária e TF-IDF + regressão logística. Separação por empresa evita que anúncios da mesma organização estejam simultaneamente no treino e no teste. O vocabulário do classificador é ajustado apenas no treino.

Este experimento **não avalia a qualidade do ranking**. Não existem rótulos humanos de relevância de candidato-vaga nesta base.

In [6]:
evaluation = evaluate_categories(snapshot['jobs'])
print('Treino:', evaluation['train_rows'], '| Teste:', evaluation['test_rows'])
print('Empresas compartilhadas:', evaluation['company_overlap'])
print('Modelo:', evaluation['model'])
print('Baseline:', evaluation['baseline'])
matrix = pd.DataFrame(evaluation['confusion_matrix'], index=evaluation['classes'], columns=evaluation['classes'])
print('\nMatriz de confusão (real nas linhas, previsão nas colunas):')
print(matrix.to_string())

Treino: 223 | Teste: 93
Empresas compartilhadas: 0
Modelo: {'accuracy': 0.9032, 'macro_f1': 0.8813}
Baseline: {'accuracy': 0.7312, 'macro_f1': 0.4224}

Matriz de confusão (real nas linhas, previsão nas colunas):
                        Dados e analytics  Engenharia de software
Dados e analytics                      22                       3
Engenharia de software                  6                      62


## 6. Conclusões e próximas hipóteses

- O painel ajuda a formular decisões de aprendizado verificáveis na amostra, sem prometer contratação.
- Um único split não estima toda a variabilidade do classificador. Uma próxima análise pode usar validação cruzada por empresa e depois um teste temporal independente.
- O silhouette baixo indica sobreposição: as fronteiras entre grupos de competências são graduais.
- Uma avaliação do ranking exigiria perfis consentidos e julgamentos humanos de relevância.
- Para medir demanda ao longo do tempo, seria necessário coletar snapshots comparáveis, deduplicar e controlar mudanças da fonte.
- O extrator pode ser ampliado com detecção de negação e distinção entre requisito obrigatório e diferencial.

Fonte: [Jobicy](https://jobicy.com/jobs-rss-feed). Métodos: [TF-IDF](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction), [K-means](https://scikit-learn.org/stable/modules/clustering.html#k-means).